# RAG Deep Dive — Part 5

## Deployment — FastAPI, Streamlit, Vercel & GitHub Actions
### A continuation of RAG Part 4 · Built on the KenteCode AI Knowledge Base · FastAPI · Streamlit · Vercel · GitHub Actions

> **Lecture Type:** Detailed Technical Lecture — for developers, ML engineers, and students in the Generative AI Engineering program.
> **Prerequisites:** RAG Part 1 (Overview), Part 2 (Chunking, Embeddings & Vector DB), Part 3 (Retrieval), and Part 4 (Evaluation & Testing). Python. A free Vercel account and a free Streamlit Community Cloud account, both connected to GitHub. An OpenAI API key and a Pinecone API key.
---

In [1]:
# Uncomment to install if needed.
# %pip install fastapi uvicorn streamlit requests

## Section 11: Where We Are & The Deployment Picture

Parts 2–4 built something real: a chunking + embedding pipeline that indexes the KenteCode AI knowledge base into Pinecone (Part 2), an `AdvancedRetriever` that combines dense search, BM25, RRF, reranking, and query rewriting (Part 3), and a RAGAS + pytest suite that proves the answers are grounded and relevant (Part 4). All of it lives in `retrieval_pipeline.py` and a couple of notebooks.

The problem: **none of that is reachable by anyone except whoever has this repo open in Jupyter.** There's no URL, no UI a non-engineer could use, no guarantee that a change someone pushes tomorrow doesn't quietly break retrieval. Part 5 closes that gap — not by building anything new algorithmically, but by wrapping what already works in the scaffolding that turns "code on my laptop" into "a product someone else can use and trust."

### The target architecture

```
        (Streamlit Community Cloud)                    (Vercel)
 Browser ──────▶ Streamlit UI  ──── HTTPS ────▶  FastAPI  ──────▶ retrieval_pipeline.py ──────▶ Pinecone + OpenAI
      (kentecode-gpt/app/streamlit_app.py)  (kentecode-gpt/api/main.py, serverless)   (AdvancedRetriever, generate_answer)
```

Two separately deployed pieces, talking over plain HTTP — not one deployed monolith. Section 12 builds the right-hand side (the API), Section 13 builds the left-hand side (the UI), Section 14 puts both on the internet, and Section 15 adds an automatic check that stops a broken change from reaching either one.

### Everything deployable lives in one folder: `kentecode-gpt/`

```
kentecode-gpt/
├── data/
│   └── KenteCode_AI_Website.pdf   # the source document — self-contained, doesn't reach outside the folder
├── indexing/
│   ├── build_index.py       # Part 2's PDF -> chunk -> embed -> upsert pipeline, as a script
│   └── requirements.txt      # scoped deps for indexing only
├── tests/
│   ├── conftest.py           # Part 4's RAGAS fixtures
│   ├── test_rag_quality.py   # Part 4's golden-set regression suite
│   ├── golden_set.json
│   └── requirements.txt      # scoped deps for testing only
├── pytest.ini                 # same `llm` marker convention as the course repo's
├── api/
│   ├── main.py              # FastAPI app
│   └── requirements.txt      # scoped deps for the Vercel build
├── app/
│   ├── streamlit_app.py      # Streamlit chat UI
│   └── requirements.txt      # scoped deps for the Streamlit Cloud build
├── .streamlit/
│   └── secrets.toml.example
├── retrieval_pipeline.py     # a synced copy — see the note below
├── vercel.json
└── .env.example
```
### What "deployment" adds that a notebook can't give you

- **A stable URL** — the API and UI exist independent of your laptop being on.
- **A UI a non-engineer can use** — nobody outside this class is going to open a Jupyter notebook to ask KenteCodeGPT a question.
- **Tests that gate changes** — Part 4 built a quality suite; Section 15 wires it into the one place it actually stops a bad change: before it ships.

### Code Block 11.1 — Reconnect to the Pipeline

In [2]:
import retrieval_pipeline as rag

print("Imported `retrieval_pipeline.py` — Part 3's retrieval pipeline, wrapped in a REST API starting in Section 12.")
print(f"  Dense index (vectors) : {rag.INDEX_NAME}")
print(f"  Sparse index (BM25)   : {rag.SPARSE_INDEX_NAME}")
print(f"  Chat model            : {rag.CHAT_MODEL}")
print(f"  Embedding model       : {rag.EMBEDDING_MODEL}")

Imported `retrieval_pipeline.py` — Part 3's retrieval pipeline, wrapped in a REST API starting in Section 12.
  Dense index (vectors) : kentecode-rag-class
  Sparse index (BM25)   : kentecode-rag-sparse-class
  Chat model            : gpt-4o-mini
  Embedding model       : text-embedding-3-small


### Code Block 11.2 — The Indexing Script (`kentecode-gpt/indexing/build_index.py`)

Same pipeline Part 2 built interactively, cell by cell, now packaged as one script: `load_and_clean_pdf` (PyMuPDFLoader + the same `clean_pdf_text` boilerplate-stripping regexes) → `chunk_document` (`RecursiveCharacterTextSplitter`, `chunk_size=500`, `chunk_overlap=50` — identical settings to Part 2's production corpus builder) → `embed_chunks` (batched `text-embedding-3-small` calls) → `upsert_dense` / `upsert_sparse` (both Pinecone indexes, same chunk ids in both, so Part 3's RRF fusion can still match a dense hit and a sparse hit as the same chunk).

### Code Block 11.3 — Run It, Twice, to Prove Idempotency

The dense and sparse indexes this whole notebook queries against were already built by an earlier run of this exact script. Running it again here should overwrite those same 13 vectors in place — not add 13 more. `cwd="kentecode-gpt"` so `build_index.py`'s own relative path to `data/KenteCode_AI_Website.pdf` resolves the same way it would from a plain `python indexing/build_index.py` run from inside that folder.

In [4]:
import subprocess
import sys

before = rag.pinecone_index_dense.describe_index_stats()["total_vector_count"]

result = subprocess.run(
    [sys.executable, "indexing/build_index.py"],
    cwd="kentecode-gpt",
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

after = rag.pinecone_index_dense.describe_index_stats()["total_vector_count"]
print(f"dense index vector count: {before} before this run -> {after} after (unchanged = idempotent)")

Indexing report
  source_documents: 1
  total_chunks: 13
  avg_chunk_size_chars: 412.1
  embedding_model: text-embedding-3-small
  embedding_dim: 1536
  dense_index: kentecode-rag-class
  dense_vectors_upserted: 13
  sparse_index: kentecode-rag-sparse-class
  sparse_records_upserted: 13



dense index vector count: 13 before this run -> 13 after (unchanged = idempotent)


## Section 12: FastAPI Backend

> ### 📖 Concept — What Is a Web API, and Why FastAPI
>
> Everything Parts 2–4 built is a Python function call: `rag.generate_answer(question)`. That's fine inside a notebook, but a browser, a phone, or another team's service can't import your Python module — they need to reach it over a network, with an address (a URL), a request format, and a response format they can rely on. That's what a **web API** is: a contract, exposed over HTTP, for calling code you don't have direct access to.
>
> **FastAPI**, concretely, is a Python web framework built on two pieces:
> - **Starlette** — the ASGI toolkit that actually handles HTTP requests/responses, routing, and middleware.
> - **Pydantic** — the validation library that turns Python type hints into runtime request/response checking.
>
> Worth placing it against what you may already know: **Flask** and **Django** are **WSGI** frameworks — synchronous, one request occupies one worker thread until it finishes. FastAPI is **ASGI** — it can hold many requests concurrently on one process, which matters here because `/query` spends most of its time *waiting* on OpenAI and Pinecone, not computing. A sync framework blocks a whole worker on that wait; an async-capable one doesn't have to.
>
> **The pieces of a FastAPI app, mapped to what Code Block 12.2 actually uses:**
> - **Path operations** — `@app.get("/health")` / `@app.post("/query")` decorators map an HTTP verb + path to a Python function.
> - **Pydantic models** — `QueryRequest`/`QueryResponse` below. FastAPI validates every incoming request against the model automatically — a malformed body never reaches your function; it becomes a `422` before your code runs at all.
> - **Dependency injection (`Depends`)** — FastAPI's mechanism for reusable setup (DB connections, auth checks) shared across routes. This lecture's API is small enough not to need it, but it's the first thing to reach for once you add e.g. an API-key check in front of `/query`.
> - **Automatic OpenAPI docs** — because routes are typed, FastAPI generates a live `/docs` page (Swagger UI) for free. Code Block 12.3 opens it.
> - **Uvicorn** — FastAPI is a framework, not a server. **Uvicorn** is the ASGI server that actually binds a port and runs your app. `uvicorn api.main:app` means "run the ASGI app object named `app`, found inside `api/main.py`" — Section 11 already put that file at `kentecode-gpt/api/main.py`, so Code Block 12.3 runs this command with `kentecode-gpt/` as the working directory.

### The `/query` contract

| Method | Path | Request | Response |
|---|---|---|---|
| `POST` | `/query` | `{ "question": str, "top_k": int? }` | `{ "answer": str, "sources": [{ "chunk_id", "title", "text", "score" }] }` |
| `GET` | `/health` | — | `{ "status": "ok" }` |

### Code Block 12.1 — Pydantic Request/Response Models

The typed contract from the table above, as Pydantic models — this is what turns "please send JSON shaped roughly like this" into something FastAPI validates automatically.

In [5]:
from typing import List, Optional
from pydantic import BaseModel, Field


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, description="The user's question.")
    top_k: Optional[int] = Field(
        default=None, ge=1, le=10,
        description="Number of cited chunks to return. Defaults to the retriever's configured top_k.",
    )


class Source(BaseModel):
    chunk_id: str
    title: str
    text: str
    score: float


class QueryResponse(BaseModel):
    answer: str
    sources: List[Source]


# A request with no "question" key, or question="", now fails validation automatically:
try:
    QueryRequest(question="")
except Exception as e:
    print("Rejected, as expected:", e)

Rejected, as expected: 1 validation error for QueryRequest
question
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


### Code Block 12.2 — The FastAPI App (`kentecode-gpt/api/main.py`)

The models above, wired into the two endpoints, with CORS and the error handling described earlier. This is a real file — `kentecode-gpt/api/main.py` — not something this notebook generates; printing it here just keeps the lecture and the file in sync.

Note how little of this is new: `query()` calls `retriever.retrieve()` and `rag.generate_answer()` — the exact same `AdvancedRetriever` and grounded-answer function Parts 3 and 4 already built and evaluated. The API's job is routing, validation, and error translation, not reimplementing retrieval. `import retrieval_pipeline as rag` inside this file resolves to `kentecode-gpt/retrieval_pipeline.py` — the packaged copy from Section 11 — because of how Code Block 12.3 launches it.

In [6]:
from pathlib import Path

print(Path("kentecode-gpt/api/main.py").read_text(encoding="utf-8"))

"""FastAPI backend for KenteCodeGPT.

Wraps `retrieval_pipeline.py` (Part 3's AdvancedRetriever + generate_answer)
behind a small REST API: `POST /query` and `GET /health`. This is the only
thing Vercel runs in production, and the only thing the Streamlit UI is
allowed to talk to — the UI never imports `retrieval_pipeline` directly.
"""
import logging
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from openai import OpenAIError
from pinecone.exceptions import PineconeApiException
from pydantic import BaseModel, Field

import retrieval_pipeline as rag

logger = logging.getLogger("kentecode_api")

app = FastAPI(title="KenteCodeGPT API", version="1.0.0")

# Streamlit runs on a different origin (localhost:8501 locally, a different
# domain once deployed to Streamlit Community Cloud), so the browser needs
# CORS headers to allow the cross-origin fetch from chat UI -> this API.
app.add_middleware(
    CORSMiddl

### Code Block 12.3 — Run Locally & Call It

`uvicorn api.main:app --reload`, run from *inside* `kentecode-gpt/`, is what you'd type in a terminal — `api.main` only resolves as a module path relative to that folder (`kentecode-gpt` itself, with a hyphen, isn't even a legal Python package name). Inside the notebook we launch it as a background subprocess with `cwd="kentecode-gpt"` to match, so the notebook can call it and show you the output, then shut it down at the end of the cell.

In [7]:
import subprocess
import sys
import time

import requests

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--port", "8000"],
    cwd="kentecode-gpt",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(20):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).ok:
            break
    except requests.ConnectionError:
        time.sleep(0.5)

print("GET /health  ->", requests.get("http://127.0.0.1:8000/health").json())

resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"question": "Can I take the AI for Everyone course if I have no coding experience?"},
)
print("POST /query  ->", resp.status_code)
answer = resp.json()
print("answer:", answer["answer"])
print(f"cited {len(answer['sources'])} sources, top score {answer['sources'][0]['score']:.3f}")

print("\nSwagger UI (open while the server is running): http://127.0.0.1:8000/docs")

server.terminate()
server.wait()

GET /health  -> {'status': 'ok'}


POST /query  -> 200
answer: Yes, you can take the AI for Everyone course if you have no coding experience. It is designed for complete beginners and requires no technical background [1].
cited 3 sources, top score 0.859

Swagger UI (open while the server is running): http://127.0.0.1:8000/docs


1

## Section 13: Streamlit Frontend

> ### 📖 Concept — Streamlit's Execution Model: Rerun, Not React
>
> **Streamlit** is a Python library that turns a plain top-to-bottom script into a web app — no HTML, CSS, or JavaScript required. That convenience comes from one design decision that explains almost every Streamlit surprise you'll hit: **every time a user interacts with a widget, Streamlit reruns your entire script from the top.**
>
> This is not how React (or most stateful GUI frameworks) work — there's no persistent component tree holding local state between interactions. A plain variable is gone the instant the script finishes, and the *next* interaction starts that script fresh:
>
> ```python
> count = 0
> if st.button("Add one"):
>     count += 1
> st.write(count)   # always prints 1 — count is reset to 0 on every rerun, every time
> ```
>
> **`st.session_state`** exists specifically to survive that: it's a dict Streamlit keeps alive *across* reruns. That's why `app/streamlit_app.py` below stores chat history in `st.session_state.messages` instead of a plain list — a plain list would evaporate the moment the user sent a second message.
>
> **Other components used in this section's app:**
> - **`st.chat_message` / `st.chat_input`** — layout primitives Streamlit added specifically for LLM chat UIs; `st.chat_input` returns the typed text only on the run where the user just submitted it, `None` on every other rerun.
> - **`st.expander`** — a collapsible container, used here to hide the cited chunks behind a click instead of dumping raw retrieved text into the main conversation.
> - **`st.sidebar`** — a persistent side panel, used here for the API base URL override.
> - **Caching (`st.cache_data` / `st.cache_resource`)** — one line to know exists: it lets a value survive reruns *without* going through session state, useful for anything expensive to (re)build (a DB client, a loaded model). This app is thin enough not to need it, but you will the moment you build something heavier on Streamlit.
>
> **Where Streamlit fits.** It's the right tool for exactly what this section builds: an internal tool, a demo, a chat prototype in front of an API. It is not what you'd reach for to ship a polished consumer product — no custom design system, and the full-script-rerun model has a real performance ceiling once a page does enough work. That's a feature here, not a gap: it's why Streamlit gets a UI shipped in one section instead of a separate frontend framework and a build pipeline.

### Why the UI never imports `retrieval_pipeline`

`app/streamlit_app.py` only ever calls `requests.post(f"{api_base_url}/query", ...)` — it does not `import retrieval_pipeline`, does not hold a Pinecone or OpenAI client, and does not know anything about `AdvancedRetriever`. Two reasons, one practical and one architectural:

1. **It has no choice, once deployed.** Section 14 puts the UI and the API on two different hosts — different processes, different filesystems. There is no `retrieval_pipeline.py` for the deployed UI to import even if it wanted to.
2. **Separation of concerns, even locally.** The UI's only job is rendering a conversation and calling an API. If retrieval logic changes, one file (`kentecode-gpt/api/main.py`, backed by `kentecode-gpt/retrieval_pipeline.py`) changes — the UI is untouched, and could be swapped for a different frontend entirely without touching a line of retrieval code.

### Code Block 13.1 — The Streamlit Chat UI (`kentecode-gpt/app/streamlit_app.py`)

Same pattern as Section 12: a real file — `kentecode-gpt/app/streamlit_app.py` — printed here to keep the lecture and the file in sync.

One config detail worth reading closely: `_get_config()` checks `st.secrets` before `os.environ`. That's not defensive-programming paranoia — Streamlit Community Cloud injects the secrets you set in its dashboard into `st.secrets`, **not** into `os.environ`, so `os.getenv("API_BASE_URL")` alone would silently return the local default forever once deployed. `st.secrets` also raises outright if no `secrets.toml` exists anywhere (the normal case for local dev without one), which is why the lookup is wrapped in a `try/except` rather than a plain dict `.get()`.

In [8]:
print(Path("kentecode-gpt/app/streamlit_app.py").read_text(encoding="utf-8"))

"""Streamlit chat UI for KenteCodeGPT.

Talks only to the FastAPI backend over HTTP (`POST /query`) — never imports
`retrieval_pipeline` or touches Pinecone/OpenAI directly. That's what makes
this deployable to a completely different host (Streamlit Community Cloud)
than the API (Vercel).
"""
import os

import requests
import streamlit as st

st.set_page_config(page_title="KenteCodeGPT", page_icon="🎓")


def _get_config(key: str, default: str) -> str:
    """Streamlit Community Cloud injects secrets.toml values into `st.secrets`,
    not into `os.environ` — so check secrets first, then fall back to a real
    env var (local dev, or Vercel-style platforms), then the default.
    `st.secrets` raises if no secrets.toml exists at all, which is the normal
    case for local development without one."""
    try:
        return st.secrets[key]
    except Exception:
        return os.getenv(key, default)


DEFAULT_API_BASE_URL = _get_config("API_BASE_URL", "http://localhost:8000")

with st.side

### Code Block 13.2 — Run Locally

Same background-subprocess trick as Code Block 12.3 — `streamlit run` normally blocks your terminal (it's a long-lived server), so we launch it, confirm it's actually listening, then shut it down. Streamlit's UI itself needs a browser to interact with; open `http://localhost:8501` yourself (with the API from 12.3 still running, or restarted) to actually chat with it.

One difference from Code Block 12.3: this one does **not** set `cwd="kentecode-gpt"`. Streamlit Community Cloud always runs your app with the working directory set to the *repository* root, regardless of which subfolder the entrypoint lives in — so testing locally the same way (repo root as `cwd`, `kentecode-gpt/app/streamlit_app.py` as the target path) is what actually matches how it behaves once deployed.

In [9]:
import subprocess
import sys
import time

import requests

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "kentecode-gpt/app/streamlit_app.py",
        "--server.headless", "true", "--server.port", "8501",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

up = False
for _ in range(20):
    try:
        up = requests.get("http://127.0.0.1:8501", timeout=1).ok
        if up:
            break
    except requests.ConnectionError:
        time.sleep(0.5)

print("Streamlit reachable on http://127.0.0.1:8501 :", up)
print("Open it in a browser, point the sidebar's API base URL at http://127.0.0.1:8000, and chat.")

streamlit_proc.terminate()
streamlit_proc.wait()

Streamlit reachable on http://127.0.0.1:8501 : True
Open it in a browser, point the sidebar's API base URL at http://127.0.0.1:8000, and chat.


1

## Section 14: Deploying — Vercel (API) + Streamlit Community Cloud (UI)

> ### 📖 Concept — What "Deployment" Actually Means
>
> Running `uvicorn` in Code Block 12.3 makes the API reachable — for as long as that cell's Python process is alive, and only on `127.0.0.1`, meaning only this machine. **Deployment** is taking that same code and making it run continuously, on infrastructure you don't manage, at a stable public URL. Two different platforms host the two halves here, for a reason that follows directly from Section 13's theory.
>
> **What is Vercel, concretely.** A platform built around **git-triggered builds**: push to GitHub, Vercel detects the change, builds it, deploys it. It supports two execution models — static/edge assets, and **serverless functions**. This lecture only needs the serverless side.
>
> **The serverless function model — the concept that matters most here.** A serverless function is **stateless** and **ephemeral**: no process sits around waiting for requests. The platform spins up an instance on demand, runs one request (or a short burst of them), and can freeze or kill that instance once traffic stops. First request after idle time pays a **cold start** — worth knowing about so a slow first response in a demo isn't a surprise.
>
> This is exactly why **Streamlit cannot be deployed to Vercel**: Section 13 established that Streamlit needs one long-lived process holding a WebSocket connection and in-memory `st.session_state`. "Spin up on demand, then die" is the opposite of that. The API is a perfect fit for serverless — each `/query` call is independent, nothing needs to persist between requests — the UI is not.
>
> **Vercel's build/deploy pipeline.** Connect the GitHub repo once; from then on, every push to a branch gets its own **preview deployment** (its own throwaway URL, useful for testing a PR before merging), and every push to `main` updates the **production deployment** at the stable URL. **Environment variables** (`OPENAI_API_KEY`, `PINECONE_API_KEY`, etc.) are set in the Vercel project dashboard and injected at runtime — never committed to the repo, same idea as `.env` locally, different mechanism because it's a different platform.
>
> **What is Streamlit Community Cloud, concretely.** Unlike Vercel, this isn't general-purpose compute — it's a managed host built specifically to run **one persistent Streamlit process per app**, which is exactly the model Section 13 required. Deploy flow: connect the GitHub repo, point at `kentecode-gpt/app/streamlit_app.py`, Cloud installs the `requirements.txt` next to it and keeps that one process running (free-tier apps sleep after a period of inactivity and wake on the next visit — worth knowing, not a bug).
>
> **The pattern repeats — name it explicitly.** Vercel env vars, Streamlit Cloud's `secrets.toml`, GitHub Actions secrets (Section 15) — three different mechanisms, one idea: secrets live in the platform that runs the code, never in the git history.

### Code Block 14.1 — `kentecode-gpt/vercel.json`

Two things this config does: tell Vercel's Python builder that `api/main.py` (relative to `kentecode-gpt/`, once that's the linked project root — Code Block 14.2 links it) is the entry point, and route every path to that one function so `/health` and `/query` both reach it. `includeFiles: "retrieval_pipeline.py"` explicitly bundles the sibling copy from Section 11 — technically closer to a belt-and-suspenders move now that it lives inside the same deployable folder (Vercel's Git-integration build already uploads everything under the linked root), but cheap insurance against a static-analysis miss on a live deploy.

`kentecode-gpt/api/requirements.txt` is a separate, deliberately smaller dependency list than the course's root `requirements.txt` — `@vercel/python` installs whatever `requirements.txt` sits next to the entry file, and the course root's pulls in `ragas`, `langgraph`, `pytest`, and the rest of this whole repo's dependencies. Installing all of that into a serverless function is slower and risks Vercel's function size limit for no benefit — the deployed API only ever imports `fastapi`, `uvicorn`, `openai`, `pinecone`, and `python-dotenv`.

In [10]:
print(Path("kentecode-gpt/vercel.json").read_text(encoding="utf-8"))
print()
print(Path("kentecode-gpt/api/requirements.txt").read_text(encoding="utf-8"))

{
  "builds": [
    {
      "src": "api/main.py",
      "use": "@vercel/python",
      "config": { "includeFiles": "retrieval_pipeline.py" }
    }
  ],
  "routes": [
    { "src": "/(.*)", "dest": "api/main.py" }
  ]
}


fastapi
uvicorn
openai
pinecone
python-dotenv



### Code Block 14.2 — Vercel: Link the Project

`vercel login` opens a browser for interactive OAuth — that doesn't work inside a notebook kernel, so this cell (and the ones through 14.4) are commented out on purpose: real commands, meant to be uncommented and run from an actual terminal, one at a time, not executed blindly by "run all cells."

Note the `cd kentecode-gpt` — `vercel link` connects *the current directory* to a Vercel project, and `kentecode-gpt/` is meant to be the whole project Vercel sees, not the course repo root. It creates a project on the first run and asks which scope/team to deploy under.

In [11]:
# Run from a real terminal, not from this notebook kernel.
# %pip install -g vercel   # or: npm i -g vercel
# !cd kentecode-gpt && vercel login
# !cd kentecode-gpt && vercel link

### Code Block 14.3 — Vercel: Configure Environment Variables & Deploy

`vercel env add` prompts for the value and which environment (`production`/`preview`/`development`) it applies to — these become the runtime environment variables Section 12 already reads via `retrieval_pipeline.py`'s `os.getenv(...)` calls, nothing in `api/main.py` changes to read them. Add every variable `kentecode-gpt/.env.example` lists locally.

`vercel --prod` deploys straight to the stable production URL from your machine; connecting the GitHub repo in the Vercel dashboard instead makes every push to `main` deploy automatically (the CD half of "CI/CD" — Section 15 covers the CI half) — pick one, they're not mutually exclusive. Either way, set the dashboard's **Root Directory** to `kentecode-gpt` if you connect the repo, so Vercel builds from the package, not the whole course monorepo.

In [12]:
# cd kentecode-gpt first, then:
# !vercel env add OPENAI_API_KEY production
# !vercel env add PINECONE_API_KEY production
# !vercel env add INDEX_NAME production
# !vercel env add SPARSE_INDEX_NAME production
# !vercel env add EMBEDDING_MODEL production
# !vercel env add EMBEDDING_DIM production
# !vercel env add CHAT_MODEL production
#
# !vercel --prod

### Code Block 14.4 — Verify the Live API

Once `vercel --prod` prints a URL, set it below (or export `VERCEL_URL` before starting Jupyter) and rerun this cell — it's the same `/health` check Code Block 12.3 ran against `localhost`, now against the deployed one.

In [13]:
import os

import requests

VERCEL_URL = os.getenv("VERCEL_URL")  # e.g. "https://kentecode-gpt.vercel.app" — set after `vercel --prod`

if VERCEL_URL:
    print("GET /health ->", requests.get(f"{VERCEL_URL}/health", timeout=10).json())
else:
    print("VERCEL_URL not set yet — deploy with Code Block 14.3, then set VERCEL_URL and rerun this cell.")

VERCEL_URL not set yet — deploy with Code Block 14.3, then set VERCEL_URL and rerun this cell.


### Code Block 14.5 — Streamlit Community Cloud: Deploy the UI

Unlike Vercel, Streamlit Community Cloud has no CLI — it's dashboard- and git-driven only, so this step really is a checklist rather than a runnable cell:

1. Push this repo to GitHub (if not already) — `git push`.
2. At [share.streamlit.io](https://share.streamlit.io), **New app** → pick the repo, branch `main`, main file path `kentecode-gpt/app/streamlit_app.py` (Community Cloud deploys from the *repository* root even though the entrypoint is nested — that's also why Code Block 13.2 tested it with the repo root as `cwd`, not `kentecode-gpt/`).
3. In the app's **Settings → Secrets**, paste:
   ```toml
   API_BASE_URL = "https://<your-project>.vercel.app"
   ```
   (`kentecode-gpt/.streamlit/secrets.toml.example` shows the same key for local testing — copy it to `kentecode-gpt/.streamlit/secrets.toml`, which is already git-ignored, to test the `st.secrets` path from Code Block 13.1 before deploying.)
4. Deploy. Streamlit Cloud installs `kentecode-gpt/app/requirements.txt` — it looks for a `requirements.txt` either at the repo root or right next to the entrypoint file, and `app/requirements.txt` is the latter — and keeps the app's one persistent process running; subsequent pushes to `main` redeploy automatically.

**Verify end-to-end:** open the Streamlit Cloud URL, ask a question, confirm the answer and cited sources render — Section 16 walks through exactly this trace.

## Section 15: GitHub Actions — CI

> ### 📖 Concept — CI/CD, and What GitHub Actions Actually Is
>
> **CI/CD** names two related but distinct automations: **Continuous Integration** — automatically running checks (tests, linting) on every code change — and **Continuous Deployment** — automatically shipping a change that passes those checks. This lecture deliberately keeps them separate: **GitHub Actions handles CI only** (runs the test suite); **Vercel and Streamlit Cloud handle their own CD** (Section 14's git-triggered deploys). Two loosely-coupled automations instead of one that does everything is a simplicity choice worth naming — Actions doesn't need to know how to deploy either platform, and either platform can be swapped without touching the workflow file.
>
> **GitHub Actions, concretely**, is GitHub's built-in automation: YAML files under `.github/workflows/`, each one a **workflow**, that GitHub runs in response to repo events. The vocabulary, mapped directly to lines you'll see in Code Block 15.1:
>
> - **Event / trigger** (`on:`) — what starts the workflow. `push`, `pull_request`, `schedule` (cron syntax), `workflow_dispatch` (a manual "Run workflow" button). This lecture's main workflow triggers on `push`/`pull_request`; Part 4 already sketched a `schedule`-triggered one for the slow, costly LLM-graded tests — Code Block 15.2 makes that a real file.
> - **Job** (`jobs:`) — a set of steps that runs on a fresh **runner** (a hosted VM — `ubuntu-latest` here). Multiple jobs in one workflow run in parallel by default.
> - **Step** (`steps:`) — one command, or a reusable **action** (`uses: actions/checkout@v4`, `uses: actions/setup-python@v5`). An action is a shareable, versioned step — the same idea as a package, but for a CI step instead of a Python import.
> - **Runner** — GitHub-hosted runners are free (within limits) ephemeral VMs, spun up per run and thrown away after; self-hosted runners exist but aren't needed here.
> - **Secrets** (`secrets.NAME`) — encrypted values set in *Settings → Secrets and variables → Actions* on the repo, exposed to a step as an environment variable, never printed in logs, never committed. The same "secrets live in the platform" pattern from Section 14, applied to CI.

### A wrinkle worth naming: this repo's tests all need a live connection

`tests/test_rag_quality.py` — Part 4's RAGAS regression suite — marks every single test `@pytest.mark.llm`, and `pytest.ini` excludes `llm`-marked tests by default (`addopts = -m "not llm"`), specifically so a normal `pytest` run doesn't spend real API tokens. That's the right call for local development. It has one consequence worth catching before it surprises you in CI: **`tests/conftest.py` imports `retrieval_pipeline` at collection time**, and that import connects to Pinecone immediately (`connect_to_index` calls `client.has_index(...)`) — before pytest has even decided which tests match the marker filter. So even a CI job that runs *zero* `llm`-marked tests still needs valid `OPENAI_API_KEY`/`PINECONE_API_KEY` secrets, just to successfully *collect* the test file.

The second consequence: with every test excluded, `pytest tests` collects 8 items, deselects all 8, and **exits with status 5** ("no tests ran") — which GitHub Actions would otherwise report as a failed job, for a reason that has nothing to do with anything being broken. Code Block 15.1 handles this explicitly (treats exit code 5 as success) rather than silently. The honest fix is adding a fast, non-LLM test — e.g. asserting `reciprocal_rank_fusion` fuses two ranked lists correctly, no network call required — at which point this workaround stops firing on its own and starts gating something real.

### Code Block 15.1 — `.github/workflows/tests.yml`

Runs on every push to `main` and every pull request. Given the wrinkle above, this job is currently closer to "prove the pipeline still imports cleanly" than "run a real regression suite" — that's an honest description of where this repo's test coverage actually is, not a limitation of the workflow itself.

In [14]:
print(Path(".github/workflows/tests.yml").read_text(encoding="utf-8"))

name: tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  pytest:
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
      PINECONE_API_KEY: ${{ secrets.PINECONE_API_KEY }}
      INDEX_NAME: ${{ secrets.INDEX_NAME }}
      SPARSE_INDEX_NAME: ${{ secrets.SPARSE_INDEX_NAME }}
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run test suite (skips tests marked "llm" — see pytest.ini)
        run: |
          pytest tests -v --junitxml=tests/pytest-results.xml
          code=$?
          # Every test in this repo is currently `llm`-marked (see tests/test_rag_quality.py),
          # so the default "not llm" filter deselects all of them — pytest's own signal for
          # that is exit code 5 ("no tests ran"), which would otherwise fail this job for a
          # rea

### Code Block 15.2 — `.github/workflows/ragas-nightly.yml`

The workflow Part 4's "Wiring This Into CI" section sketched as YAML in a markdown cell — here it is as a real file. Runs the `llm`-marked RAGAS suite nightly (`schedule`) and on-demand (`workflow_dispatch`), using the same repo secrets as the fast workflow, and uploads the JUnit XML as a build artifact so a failure is inspectable after the log scrolls away.

In [15]:
print(Path(".github/workflows/ragas-nightly.yml").read_text(encoding="utf-8"))

name: rag-quality

on:
  schedule:
    - cron: '0 6 * * *'   # nightly
  workflow_dispatch: {}    # or trigger manually before a release

jobs:
  ragas-regression:
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
      PINECONE_API_KEY: ${{ secrets.PINECONE_API_KEY }}
      INDEX_NAME: ${{ secrets.INDEX_NAME }}
      SPARSE_INDEX_NAME: ${{ secrets.SPARSE_INDEX_NAME }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: pytest tests -v -m llm --junitxml=tests/pytest-results.xml
      - uses: actions/upload-artifact@v4
        if: always()
        with:
          name: ragas-results
          path: tests/pytest-results.xml



### Setting up the secrets these workflows need

*Settings → Secrets and variables → Actions → New repository secret*, on GitHub, once per repo: `OPENAI_API_KEY`, `PINECONE_API_KEY`, `INDEX_NAME`, `SPARSE_INDEX_NAME` (and any other variable `retrieval_pipeline.py` reads that isn't already defaulted). Both workflows above reference them via `${{ secrets.NAME }}` — the same values as `.env` locally and the Vercel project's environment variables, set a third time because GitHub Actions runners don't share state with either.

### `kentecode-gpt/` has its own copy of Part 4's RAGAS suite too

Same self-containment reason as `retrieval_pipeline.py` in Section 11: `kentecode-gpt/tests/` (`conftest.py`, `test_rag_quality.py`, `golden_set.json`) and `kentecode-gpt/pytest.ini` are copies of Part 4's suite, adapted only to import the **packaged** `retrieval_pipeline.py` — one directory up from `tests/`, same relative layout as the course repo, so `conftest.py` didn't need to change beyond a docstring. Running it validates the actual code the deployed API ships, not just the pedagogical copy Parts 3–4 iterate on.

Running the full `pytest tests -m llm` suite from inside `kentecode-gpt/` reproduces Part 4's own result almost exactly: 7 of 8 golden-set questions pass, and the one that doesn't is the same `context_recall=0.0` case Part 4's "Case Study" section already walks through in depth — not a regression introduced by copying the suite, the same known characteristic of that question, in a different folder. That theory isn't repeated here; Part 4 is still the place for it.

The cell below only runs the **default**, fast `pytest tests` — the same `-m "not llm"` / exit-code-5 situation as the course repo's own suite (Section 15's "wrinkle," above), proving the suite is wired correctly without spending real API tokens every time this notebook re-executes. The `-m llm` run is worth doing yourself, from a terminal, at least once.

The course repo's own `.github/workflows/tests.yml` and `ragas-nightly.yml` still only test the root `tests/` — they don't reach into `kentecode-gpt/`. That's fine; they were never meant to. `kentecode-gpt/` has its own `.github/workflows/` for exactly that reason.

In [16]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-v", "--junitxml=tests/pytest-results.xml"],
    cwd="kentecode-gpt",
    capture_output=True, text=True,
)
print(result.stdout)
code = result.returncode
if code == 5:
    print("(exit 5 = no tests ran, because every test here is `llm`-marked — same situation as the course repo's suite)")
elif code != 0:
    print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.13.11, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\vince\Documents\repo\gen-ai\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\vince\Documents\repo\gen-ai\kentecode-gpt
configfile: pytest.ini
plugins: anyio-4.13.0, langsmith-0.9.3
collecting ... collected 8 items / 8 deselected / 0 selected

- generated xml file: C:\Users\vince\Documents\repo\gen-ai\kentecode-gpt\tests\pytest-results.xml -
============================ 8 deselected in 0.02s ============================

(exit 5 = no tests ran, because every test here is `llm`-marked — same situation as the course repo's suite)


### `kentecode-gpt/.github/workflows/` — dormant until the folder becomes its own repo

GitHub only ever reads `.github/workflows/` at a repository's **true root** — that's what made the course repo's workflows unreachable from inside `kentecode-gpt/` in the first place (Section 15's opening note). It cuts both ways: a `.github/workflows/` folder *inside* `kentecode-gpt/` is just as invisible to GitHub right now, sitting one level too deep, inert.

That stops being true the day `kentecode-gpt/` is pushed as its own repository — the plan for this package, not a hypothetical. The moment `kentecode-gpt/` *is* the repo root, `kentecode-gpt/.github/workflows/tests.yml` and `ragas-nightly.yml` start running exactly like the course repo's copies do today, with zero edits needed at extraction time. That's the actual point of building them now rather than after: prove the paths are right while they're still easy to test, not after a `git filter-repo` when nobody's looking closely at whether `tests/requirements.txt` resolves correctly.

Two differences from the course-repo versions, both because `kentecode-gpt/` won't have a single monolithic `requirements.txt` once it's standalone:
- `pip install -r requirements.txt` becomes `pip install -r tests/requirements.txt` — the scoped file from Section 15 already has everything a test run needs (`pytest`, `openai`, `pinecone`, `python-dotenv`, `ragas`, `langchain-openai`), same reasoning as `api/requirements.txt` staying lean for Vercel.
- No `cd kentecode-gpt` or `working-directory:` anywhere — paths like `tests/requirements.txt` and `tests` are already correct once this *is* the repo root; adding a redundant `cd` would be the bug, not the fix, the day this file starts actually running.

In [17]:
print(Path("kentecode-gpt/.github/workflows/tests.yml").read_text(encoding="utf-8"))
print()
print(Path("kentecode-gpt/.github/workflows/ragas-nightly.yml").read_text(encoding="utf-8"))

name: tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  pytest:
    runs-on: ubuntu-latest
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
      PINECONE_API_KEY: ${{ secrets.PINECONE_API_KEY }}
      INDEX_NAME: ${{ secrets.INDEX_NAME }}
      SPARSE_INDEX_NAME: ${{ secrets.SPARSE_INDEX_NAME }}
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install -r tests/requirements.txt

      - name: Run test suite (skips tests marked "llm" — see pytest.ini)
        run: |
          pytest tests -v --junitxml=tests/pytest-results.xml
          code=$?
          # Every test in this repo is currently `llm`-marked (see tests/test_rag_quality.py),
          # so the default "not llm" filter deselects all of them — pytest's own signal for
          # that is exit code 5 ("no tests ran"), which would otherwise fail this job for a
         

## Section 16: Putting It All Together

### One request, traced end to end

1. A user opens the **Streamlit Community Cloud** URL and types a question into `st.chat_input`.
2. The script reruns (Section 13's rerun model); the new input is captured, appended to `st.session_state.messages`, and `requests.post(f"{api_base_url}/query", ...)` fires — `api_base_url` is the deployed **Vercel** URL, set once in Streamlit Cloud's secrets.
3. Vercel's serverless runtime spins up (or reuses a warm) instance of `kentecode-gpt/api/main.py`. FastAPI validates the request body against `QueryRequest` — an empty question never gets this far.
4. `query()` calls `rag.retriever.retrieve(...)` — Part 3's full pipeline: rewrite → multi-query → hybrid search → RRF → rerank — against the live Pinecone indexes.
5. `rag.generate_answer(...)` builds the grounded-generation prompt from the retrieved chunks and calls OpenAI; the response is a citation-bearing answer plus the source chunks.
6. FastAPI serializes the result against `QueryResponse` and returns JSON; the Vercel function instance is free to go idle again.
7. Streamlit renders the answer in the chat, and the sources in the `st.expander` — same request/response shapes Section 12's contract table defined at the very start.

### Repo structure recap

```
kentecode-gpt/                # self-contained deployable package (Section 11) -> future repo root
├── .github/
│   └── workflows/
│       ├── tests.yml           # dormant copy (Section 15) -> activates once kentecode-gpt/ is its own repo
│       └── ragas-nightly.yml
├── data/
│   └── KenteCode_AI_Website.pdf   # source document — self-contained, doesn't reach outside the folder
├── indexing/
│   ├── build_index.py          # Part 2's pipeline as a script (Section 11)  -> builds/refreshes both Pinecone indexes
│   └── requirements.txt        # scoped deps for indexing only
├── tests/
│   ├── conftest.py            # Part 4's RAGAS fixtures (Section 15)         -> validates THIS copy of retrieval_pipeline.py
│   ├── test_rag_quality.py
│   ├── golden_set.json
│   └── requirements.txt        # scoped deps for testing only
├── pytest.ini                  # same `llm` marker convention as the course repo's
├── api/
│   ├── main.py                # FastAPI app (Section 12)             -> deployed to Vercel
│   └── requirements.txt        # scoped deps for the Vercel build
├── app/
│   ├── streamlit_app.py        # Streamlit chat UI (Section 13)       -> deployed to Streamlit Community Cloud
│   └── requirements.txt        # scoped deps for the Streamlit Cloud build
├── .streamlit/
│   └── secrets.toml.example    # template for local secrets.toml and Streamlit Cloud's Secrets panel
├── retrieval_pipeline.py       # synced copy — the deployed package's only dependency on Part 3/4's logic
├── vercel.json                 # Vercel routing/build config (Section 14)
└── .env.example

.github/
  workflows/
    tests.yml           # CI: fast suite on every push/PR (Section 15)      -> tests the course repo, not kentecode-gpt/
    ragas-nightly.yml   # CI: RAGAS regression, nightly + manual (Section 15)

retrieval_pipeline.py    # the repo-root original — Parts 3/4 and tests/ import this one, unchanged
requirements.txt         # full dev environment (this notebook + tests + the whole course)
```

`.github/workflows/` only ever runs from a repository's true root — which is why there are **two** copies here, not one shared copy. The course repo's `.github/workflows/` (top-level, active) tests the root `tests/`; `kentecode-gpt/.github/workflows/` (Section 15) tests `kentecode-gpt/tests/` and sits dormant until `kentecode-gpt/` is extracted into its own repository, at which point it becomes the active one with no edits required. Everything *deployed*, meanwhile, already lives inside `kentecode-gpt/` and nowhere else — that folder is what you'd hand someone (or point Vercel/Streamlit Cloud at, or `git push` as a new repo) today. This is the same layout `project1-rag.ipynb` already asks you to build for your own corpus — everything in this section is the reference implementation for that assignment, not just a demo.

### What's next

`project1-rag.ipynb` is the capstone: repeat Parts 1–5 end to end, on a document set of your own choosing, and ship it exactly like this — a FastAPI backend on Vercel, a Streamlit UI on Streamlit Community Cloud, tests wired into GitHub Actions.

## Recap: What Part 5 Added

| # | What | Where it lives |
|---|---|---|
| 1 | A self-contained deployable package, separate from the course monorepo | `kentecode-gpt/` |
| 2 | Part 2's indexing pipeline, packaged as an idempotent script | `kentecode-gpt/indexing/build_index.py` |
| 3 | A typed REST API around `retrieval_pipeline.py` — `/query`, `/health`, Pydantic validation, CORS, translated error codes | `kentecode-gpt/api/main.py` |
| 4 | A chat UI that talks only to that API | `kentecode-gpt/app/streamlit_app.py` |
| 5 | A live, public API on Vercel (serverless, git-triggered deploys) | Vercel project |
| 6 | A live, public UI on Streamlit Community Cloud | Streamlit Cloud app |
| 7 | An automatic test gate on every push/PR, plus a nightly RAGAS regression job — for the course repo | `.github/workflows/` |
| 8 | Part 4's RAGAS suite, packaged to validate the deployed copy of the pipeline | `kentecode-gpt/tests/` |
| 9 | The same CI, pre-wired for `kentecode-gpt/` — dormant until it's its own repo | `kentecode-gpt/.github/workflows/` |

### Series recap: Parts 1 → 5

| Part | Question it answers |
|---|---|
| 1 — Overview | What is RAG, and why not just fine-tune or stuff everything in the prompt? |
| 2 — Chunking, Embeddings & Vector DB | How do you turn documents into something searchable? |
| 3 — Retrieval | How do you find the *right* chunks — dense, sparse, hybrid, reranked, rewritten? |
| 4 — Evaluation | How do you know any of it is actually working? |
| 5 — Deployment | How does anyone other than you actually use it? |

That's the whole path from a folder of PDFs to a URL someone else can open and trust.